# 🛰️🔆 TKG Solar — Colab GPU Training (clone & run)

Tái lập huấn luyện **TKGSolarModel** (M1→M10) end-to-end *trực tiếp trên notebook*:
clone repo → cài deps → mount dữ liệu Drive → M1 pipeline → dựng model → **train** →
đánh giá → vẽ biểu đồ → lưu checkpoint. Mỗi bước phân tách bằng đường kẻ + công thức.

> **Trước khi chạy:** `Runtime → Change runtime type → GPU` (T4/A100). Chạy tuần tự từ trên xuống.

> ⚠️ Tái lập **cơ học/mã nguồn** (3 nguồn dữ liệu không cùng vị trí địa lý — faithful theo
> paper). "Aligned" = canh đồng hồ UTC, không phải ghép cặp vật lý. Xem `docs/assumptions.md`.

**Luồng:**
```
clone ─▶ deps ─▶ Drive data ─▶ M1(load→align→clean→split→clip→scale→window) ─▶ model ─▶ fit ─▶ eval ─▶ plot ─▶ save
```

<hr style="border:none;border-top:3px solid #d95f0e;margin:8px 0">
## 1 · Kiểm tra GPU

In [ ]:
!nvidia-smi

<hr style="border:none;border-top:3px solid #d95f0e;margin:8px 0">
## 2 · Lấy code (clone nhánh `init-project`)

Điền `GITHUB_USER` của bạn. Repo: **`tkg-solar-satellite-reasoning`**, nhánh `init-project`.

In [ ]:
import os, sys, pathlib

GITHUB_USER = 'YOUR_GITHUB_USER'   # <-- ĐIỀN username của bạn
REPO        = 'tkg-solar-satellite-reasoning'
BRANCH      = 'init-project'

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not pathlib.Path(REPO).exists():
    !git clone --branch {BRANCH} https://github.com/{GITHUB_USER}/{REPO}.git

ROOT = pathlib.Path(REPO) if pathlib.Path(REPO).exists() else pathlib.Path.cwd()
if (ROOT / 'src').exists() is False and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
print('Working dir:', pathlib.Path.cwd())

<hr style="border:none;border-top:3px solid #d95f0e;margin:8px 0">
## 3 · Cài dependencies

Colab đã có **torch CUDA** sẵn — chỉ cài thêm package còn thiếu (KHÔNG dùng `uv sync`).

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
!pip install -q timm torch_geometric h5py scikit-learn pyyaml tqdm

<hr style="border:none;border-top:3px solid #d95f0e;margin:8px 0">
## 4 · Dữ liệu (mount Google Drive)

Upload 3 nguồn lên Drive theo cấu trúc dưới, rồi điền `DATA_ROOT`:
```
<DATA_ROOT>/
  opsd/time_series_15min_singleindex.csv
  nsrdb/vietnam_2016.h5
  himawari/frames.h5
```

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# <-- ĐIỀN đường dẫn thư mục dữ liệu trên Drive của bạn:
DATA_ROOT = '/content/drive/MyDrive/tkg-solar-data'

opsd_path    = f'{DATA_ROOT}/opsd/time_series_15min_singleindex.csv'
nsrdb_path   = f'{DATA_ROOT}/nsrdb/vietnam_2016.h5'
himawari_dir = f'{DATA_ROOT}/himawari'

for p in (opsd_path, nsrdb_path, himawari_dir):
    print(('OK  ' if pathlib.Path(p).exists() else 'MISSING '), p)

<hr style="border:none;border-top:3px solid #d95f0e;margin:8px 0">
## 5 · Config (paper) + thiết bị

Nạp `paper_config.yaml` (ViT-B/16, 200 epochs, Table 5) → đặt `device=cuda`.

> ⚠️ **min_steps.** Paper config dùng `min_steps=2000` (cần ~tuần dữ liệu). Nếu bạn chỉ
> upload cửa sổ Himawari 3 ngày (~184 bước aligned) thì **hạ `min_steps`** xuống ~150,
> nếu không cổng giao nhau (Bước 6) sẽ báo `EmptyOverlapError`.

In [ ]:
from src.training.config import Config
from src.common.seeding import seed_everything
from src.common.shapes import EMBED_DIM, FUSION_DIM, HORIZON_MINUTES, N_HORIZONS

cfg = Config.from_yaml('configs/paper_config.yaml')
cfg.device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- override khi cần (mở comment nếu dataset ngắn) ---
# cfg.min_steps = 150          # data Himawari 3 ngày
# cfg.epochs = 200; cfg.batch_size = 64

seed_everything(cfg.seed)
print('device     :', cfg.device)
print('backbone   :', cfg.sat_backbone, '| img', cfg.img_size)
print('epochs     :', cfg.epochs, '| batch', cfg.batch_size, '| lr', cfg.lr)
print('min_steps  :', cfg.min_steps, '| k', cfg.k, '| embed', cfg.embed_dim)

<hr style="border:none;border-top:3px solid #d95f0e;margin:8px 0">
## 6 · M1 data pipeline

`DataPipeline.load` chạy đúng chuỗi: **load → align(10-min grid) → clean → split(70/15/15)
→ clip(μ±5σ train-only) → scale(Min-Max train-only) → window → DataLoader**. Chi tiết +
công thức từng bước xem `notebooks/data_pipeline_walkthrough.ipynb`.

Cửa sổ bắt đầu hợp lệ: $i \in \{0,\dots,T-k-\max_h h\}$; target tại $(i+k-1)+h$, $h\in(1,3,6)$.

In [ ]:
from src.data_pipeline import DataPipeline

splits = DataPipeline.load(
    opsd_path, nsrdb_path, himawari_dir,
    k=cfg.k, batch_size=cfg.batch_size, img_size=cfg.img_size,
    min_steps=cfg.min_steps, train_frac=cfg.train_frac, val_frac=cfg.val_frac,
    cache_dir=cfg.cache_dir, scaler_out=cfg.scaler_out,
)
for k_, v_ in splits.meta.items():
    print(f'  {k_:18s}: {v_}')

<hr style="border:none;border-top:3px solid #d95f0e;margin:8px 0">
## 7 · Dựng model (TKGSolarModel)

Hợp đồng chiều (paper Table 5): mọi encoder xuất **128-dim**; fusion nối 3 nhánh:

$$\mathbf{z} = [\,F_{sat}\ \Vert\ H_{met}\ \Vert\ H_{graph}\,] \in \mathbb{R}^{384},\qquad \hat{\mathbf{y}} = \text{MLP}(\mathbf{z}) \in \mathbb{R}^{3}$$

trong đó $F_{sat}$ = ViT vệ tinh, $H_{met}$ = GRU+Transformer khí tượng, $H_{graph}$ = GAT+Fourier trên TKG động.

In [ ]:
from src.fusion_predictor.tkg_solar_model import TKGSolarModel

model = TKGSolarModel.from_config(cfg)
n_params = sum(p.numel() for p in model.parameters())
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'FUSION_DIM = {FUSION_DIM} = 3 x {EMBED_DIM}')
print(f'tham số    : {n_params:,} (trainable {n_train:,})')

<hr style="border:none;border-top:3px solid #d95f0e;margin:8px 0">
## 8 · Huấn luyện (train)

`fit` (`src/training/train_loop.py`): Adam + **MSE trên target đã scale**; early-stop
theo **val-MAE ở đơn vị gốc** (inverse-transform); lưu checkpoint tốt nhất; grad-clip.

**Hàm mất mát** (scaled, $B$ batch, $H{=}3$ horizon):
$$\mathcal{L} = \frac{1}{B\,H}\sum_{b=1}^{B}\sum_{h=1}^{H}\big(\hat{y}'_{b,h} - y'_{b,h}\big)^2$$

**Tiêu chí early-stop** (đơn vị gốc, qua inverse-scale $\text{inv}(\cdot)$):
$$\text{val-MAE} = \frac{1}{N\,H}\sum_{n,h}\big|\,\text{inv}(\hat{y}'_{n,h}) - \text{inv}(y'_{n,h})\,\big|$$

In [ ]:
from src.training.train_loop import fit

history = fit(model, splits, cfg, verbose=True)
print('\nbest val-MAE :', round(history['best_val_mae'], 5))
print('checkpoint   :', history['best_checkpoint'])

<hr style="border:none;border-top:3px solid #d95f0e;margin:8px 0">
## 9 · Đường học (train loss & val MAE)

In [ ]:
import matplotlib.pyplot as plt
ep = range(1, len(history['train_loss'])+1)
fig, ax1 = plt.subplots(figsize=(8,4))
ax1.plot(ep, history['train_loss'], 'o-', color='#d95f0e', label='train loss (MSE, scaled)')
ax1.set_xlabel('epoch'); ax1.set_ylabel('train loss', color='#d95f0e')
ax2 = ax1.twinx()
ax2.plot(ep, history['val_mae'], 's-', color='#2c7fb8', label='val MAE (orig units)')
ax2.set_ylabel('val MAE', color='#2c7fb8')
plt.title('Learning curve'); fig.tight_layout(); plt.show()

<hr style="border:none;border-top:3px solid #d95f0e;margin:8px 0">
## 10 · Đánh giá test (MAE / RMSE / MAPE theo horizon)

Metric tính ở **đơn vị gốc** (`src/metrics/`). MAPE chỉ trên mẫu ban ngày
$|y|\ge\texttt{mape\_min\_value}$ (PV ban đêm ≈ 0 → MAPE bùng nổ; loại có báo tỉ lệ).

$$\text{MAE}=\tfrac1N\sum|y-\hat y|,\quad \text{RMSE}=\sqrt{\tfrac1N\sum(y-\hat y)^2},\quad \text{MAPE}=\tfrac{100}{|\mathcal D|}\!\!\sum_{n\in\mathcal D}\!\Big|\tfrac{y_n-\hat y_n}{y_n}\Big|$$

In [ ]:
from src.training.train_loop import predict_loader
from src.metrics.regression_metrics import compute_per_horizon

yt, yp = predict_loader(model, splits.test_loader, cfg.device)
yt = splits.scalers.inverse_pv(yt.numpy()); yp = splits.scalers.inverse_pv(yp.numpy())
per = compute_per_horizon(yt, yp, HORIZON_MINUTES, cfg.mape_min_value)

print(f"{'horizon':>10} | {'MAE':>9} | {'RMSE':>9} | {'MAPE %':>9}")
print('-'*46)
for lab in ['overall', *[f'{m}min' for m in HORIZON_MINUTES]]:
    m = per[lab]
    print(f"{lab:>10} | {m['mae']:9.4f} | {m['rmse']:9.4f} | {m['mape']:9.2f}")

<hr style="border:none;border-top:3px solid #d95f0e;margin:8px 0">
## 11 · Dự báo vs thực tế (mỗi horizon)

In [ ]:
import numpy as np
fig, axes = plt.subplots(1, N_HORIZONS, figsize=(15, 4))
for j, mins in enumerate(HORIZON_MINUTES):
    ax = axes[j]
    ax.plot(yt[:, j], label='thực tế', lw=1.2)
    ax.plot(yp[:, j], label='dự báo', lw=1.0, alpha=.8)
    ax.set_title(f'+{mins} phút'); ax.set_xlabel('mẫu test')
    if j == 0: ax.legend()
plt.tight_layout(); plt.show()

<hr style="border:none;border-top:3px solid #d95f0e;margin:8px 0">
## 12 · Lưu checkpoint + scaler về Drive (tùy chọn)

`fit` đã lưu best checkpoint cục bộ; copy sang Drive để giữ lại sau khi runtime tắt.

In [ ]:
import shutil
SAVE_DIR = f'{DATA_ROOT}/outputs'
if IN_COLAB:
    pathlib.Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
    shutil.copy(history['best_checkpoint'], SAVE_DIR)
    if pathlib.Path(cfg.scaler_out).exists():
        shutil.copy(cfg.scaler_out, SAVE_DIR)
    cfg.save(f'{SAVE_DIR}/resolved_config.yaml')
    print('Đã lưu vào', SAVE_DIR)
else:
    print('Bỏ qua (không ở Colab). Checkpoint:', history['best_checkpoint'])

<hr style="border:none;border-top:3px solid #d95f0e;margin:8px 0">
### ✅ Hoàn tất

| Giai đoạn | Module | Đầu ra |
|---|---|---|
| M1 pipeline | `src/data_pipeline/` | DataLoaders train/val/test + scalers |
| Model | `src/fusion_predictor/` | TKGSolarModel (3×128 → MLP → 3) |
| Train | `src/training/` | best checkpoint (early-stop val-MAE) |
| Eval | `src/metrics/` | MAE/RMSE/MAPE @ 10/30/60 phút |

Tăng quy mô: bỏ comment override `epochs`/`batch_size` ở Bước 5; dùng dữ liệu dài hơn
để thoả `min_steps=2000` của paper config; bật `use_advanced_loss` (M10) sau khi MSE ổn.